# Reranking lexical search results (Wikipedia)

## Overview
Wikipedia's lexical search returns results based on keyword matching, which may not always prioritize the most relevant content for a user's specific query. Reranking uses semantic understanding to reorder search results based on relevance to the query.

This tutorial demonstrates how to search Wikipedia and improve result relevance using Cohere's rerank API.

By the end, you will have implemented a system that searches Wikipedia and reranks results for better relevance.

## What we'll cover
- Implementing Wikipedia lexical search
- Reranking results with Cohere
- Evaluating reranked search results

In [24]:
import cohere
import os
co = cohere.ClientV2(os.getenv("COHERE_API_KEY"))

In [25]:
import urllib
import requests
import wikipediaapi

## Lexical search (Wikipedia)

In [26]:
def search_wikipedia(query, max_results=20):
    """
    Modern Wikipedia search using wikipedia-api library.
    This approach is more reliable and handles API requirements automatically.
    """
    # Initialize Wikipedia API with proper user agent
    wiki_wiki = wikipediaapi.Wikipedia(
        user_agent='CohereRerankingTutorial/1.0 (Educational)',
        language='en',
        extract_format=wikipediaapi.ExtractFormat.WIKI
    )
    
    # Use the standard search API with proper headers
    search_url = "https://en.wikipedia.org/w/api.php"
    search_params = {
        'action': 'query',
        'list': 'search',
        'srsearch': query,
        'srlimit': max_results,
        'format': 'json',
        'srprop': 'snippet'
    }
    
    headers = {
        'User-Agent': 'CohereRerankingTutorial/1.0 (Educational Purpose)'
    }
    
    try:
        response = requests.get(search_url, params=search_params, headers=headers)
        response.raise_for_status()
        search_results = response.json()
        
        if 'query' not in search_results or 'search' not in search_results['query']:
            print(f"No results found for query: {query}")
            return []
        
        search_items = search_results['query']['search']
        
        if not search_items:
            print(f"No results found for query: {query}")
            return []
        
    except requests.exceptions.RequestException as e:
        print(f"Error searching Wikipedia: {e}")
        return []
    except (ValueError, KeyError) as e:
        print(f"Error parsing search results: {e}")
        return []
    
    # Fetch page details for each result
    documents = []
    for item in search_items:
        title = item['title']
        page = wiki_wiki.page(title)
        
        if page.exists():
            # Get summary (first 1200 characters)
            summary = page.summary[:1200] if len(page.summary) > 1200 else page.summary
            
            documents.append({
                "title": page.title,
                "url": page.fullurl,
                "text": summary
            })
    
    return documents

In [27]:
def print_first_n_documents(documents, num=3):
    for document in documents[:num]:
        print(document, "\n")

In [28]:
query = "where are monet's water lilies"

documents = search_wikipedia(query)

# Print the first few documents
print_first_n_documents(documents)

{'title': 'List of paintings by Claude Monet', 'url': 'https://en.wikipedia.org/wiki/List_of_paintings_by_Claude_Monet', 'text': 'This is a list of works by Claude Monet (1840–1926), including all the extant finished paintings but excluding the Water Lilies, which can be found here, and preparatory black and white sketches.\nMonet was a founder of French impressionist painting, and the most consistent and prolific practitioner of the movement\'s philosophy of expressing one\'s perceptions before nature, especially as applied to plein-air landscape painting. The term Impressionism is derived from the title of his painting Impression, Sunrise (Impression, soleil levant).\nWhat made Monet different from the other Impressionist painters was his innovative idea of creating Series paintings devoted to paintings of a single theme or subject. With the repetitious study of the subject at different times of day Monet\'s paintings show the effects of sunlight, time and weather through color and c

## Reranking

In [19]:
import yaml

top_n = 3
yaml_docs = [yaml.dump(doc, sort_keys=False) for doc in documents]

results = co.rerank(
    model="rerank-v3.5", 
    query=query, 
    documents=yaml_docs,
    top_n=top_n
)

In [25]:
# print(yaml_docs[0])

In [20]:
# Display the reranking results
text_max_chars = 500
def return_results(results, documents):
    for idx, result in enumerate(results.results):
        print(f"Rank: {idx+1}")
        print(f"Score: {result.relevance_score}")
        print(f"Title: {documents[result.index]['title']}")
        print(f"Text: {documents[result.index]['text'][:text_max_chars]}...\n")

In [21]:
return_results(results, documents)

Rank: 1
Score: 0.7657514
Title: Fondation Monet in Giverny
Text: The Fondation Claude Monet  is a nonprofit that manages the house and gardens of Claude Monet in Giverny, France, where Monet lived and painted for 43 years. Monet was inspired by his gardens, and spent years transforming them, planting thousands of flowers. He believed that it was important to surround himself with nature and paint outdoors. He created many paintings of his house and gardens, especially of water lilies in the pond, the Japanese bridge, and a weeping willow tree.
With a total of...

Rank: 2
Score: 0.60245985
Title: List of paintings by Claude Monet
Text: This is a list of works by Claude Monet (1840–1926), including all the extant finished paintings but excluding the Water Lilies, which can be found here, and preparatory black and white sketches.
Monet was a founder of French impressionist painting, and the most consistent and prolific practitioner of the movement's philosophy of expressing one's percepti

## Another example

In [22]:
query = "David Bowie hits"

documents = search_wikipedia(query)

print_first_n_documents(documents)

{'title': 'David Bowie discography', 'url': 'https://en.wikipedia.org/wiki/David_Bowie_discography', 'text': 'During his lifetime, the English singer-songwriter David Bowie (1947–2016) released 26 studio albums, nine live albums, two soundtrack albums, 26 compilation albums, eight extended plays, 128 singles and six box sets. Since his death, one further studio album, 13 live albums, one soundtrack album, one compilation album, four extended plays and six box sets have been released. Bowie also released 28 video albums and 72 music videos. Throughout his lifetime, Bowie sold at least 100 million records worldwide. In 2012, Bowie was ranked ninth best selling singles artist in United Kingdom with 10.6 million singles sold. As of January 2016, 12.09 million Bowie singles had been sold in Britain. In a period of 24 months since his death, five million records were sold in the UK, 3.1 million singles and two million albums.\nBowie\'s debut release was the 1964 single "Liza Jane" by Davie J

In [23]:
yaml_docs = [yaml.dump(doc, sort_keys=False) for doc in documents]

results = co.rerank(
    model="rerank-v3.5", 
    query=query, 
    documents=documents,
    top_n=top_n
)

return_results(results, documents)

Rank: 1
Score: 0.75268626
Title: Best of Bowie
Text: Best of Bowie is a greatest hits album by English recording artist David Bowie. Released in October 2002, four months after the critical and commercial success of the Heathen album, the songs range from his second album, David Bowie (1969) to Heathen (2002). A DVD, also titled Best of Bowie, was released too.
Initially peaking at number 11 on the UK Albums Chart upon release in October 2002, the album entered the top 10 for the first time in January 2016, after Bowie's death, and reached a new p...

Rank: 2
Score: 0.7197191
Title: Legacy (The Very Best of David Bowie)
Text: Legacy (The Very Best of David Bowie) (also known as Legacy) is a greatest hits album by English musician David Bowie, released on 11 November 2016 through Sony Music Entertainment in the US and Warner Music Group in the UK and several territories. Legacy marks Bowie’s first title to reach 400 weeks on the UK albums chart and is one of the albums which have spent 

## Conclusion
This tutorial covered searching Wikipedia using lexical search and improving result relevance through reranking with Cohere's API. The reranked results showed better alignment with query intent compared to the original Wikipedia ranking.